Black-box evaluation sources:
1. Existing CSV with input and actual output
2. API response from an external application
3. Manually prepared test cases

White-box / tracing evaluation sources:
1. Application function that we can run directly
2. Internal retriever / generator / tool functions
3. Trace data captured using @observe and update_current_trace

Simple difference:
Black-box = we already have the final output.
White-box / tracing = DeepEval watches the application while it runs.

In [1]:
!pip install deepeval groq pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.6/567.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.27.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [2]:
import os
import pandas as pd
from google.colab import userdata
from groq import Groq

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import DeepEvalBaseLLM

How DeepEval calls this Groq wrapper

We create GroqDeepEvalModel because DeepEval metrics need a model object in a fixed format.

The methods are called like this:

__init__() → Python calls automatically when groq_model = GroqDeepEvalModel() runs

load_model() → DeepEval may call this to get the actual Groq client

generate(prompt) → DeepEval metric calls this when it needs Groq to judge a test case

a_generate() → DeepEval calls this if async evaluation is used

get_model_name() → DeepEval calls this to record/display the judge model name

Where does prompt come from?

DeepEval metric creates the prompt internally.

Example: AnswerRelevancyMetric creates a judging prompt using input and actual_output.

Then DeepEval sends that prompt into generate(prompt).

Where does the response go?

Groq returns JSON response.

generate() returns that JSON back to DeepEval metric.

DeepEval reads that JSON and creates score and reason.

Simple flow:

evaluate()
→ DeepEval metric
→ metric creates prompt
→ groq_model.generate(prompt)
→ Groq API
→ JSON response
→ DeepEval metric
→ score/reason

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


"""
### Why we create `GroqDeepEvalModel` class

DeepEval metrics need a model object in DeepEval’s expected format.

That model object must have these method names:

generate()
a_generate()
get_model_name()

Why these methods are needed:

generate()        → DeepEval sends the metric prompt and gets judge response
a_generate()      → Same as generate(), but used for async evaluation
get_model_name()  → DeepEval records/displays which model judged the result

These method names should stay the same because DeepEval calls them internally.

"""
class GroqDeepEvalModel(DeepEvalBaseLLM):

    def __init__(self, model_name="openai/gpt-oss-20b"):
        # Store the Groq model name.
        self.model_name = model_name

        # Create Groq client using API key.
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        # Return Groq client to DeepEval.
        return self.client

    def generate(self, prompt: str, **kwargs) -> str:
        # DeepEval creates the metric prompt.
        # This method sends that prompt to Groq.
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        # Return Groq answer back to DeepEval.
        return response.choices[0].message.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        # Async version required by DeepEval.
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        # Tell DeepEval which model is used.
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq evaluator connected.")

Groq evaluator connected.


In [4]:
source_type = "manual"   # options: "csv", "api", "manual"


if source_type == "csv":
    from google.colab import files

    uploaded = files.upload()
    csv_file_name = list(uploaded.keys())[0]

    qa_df = pd.read_csv(csv_file_name)


elif source_type == "api":
    import requests

    api_url = userdata.get("EVALUATION_API_URL")
    api_key = userdata.get("EVALUATION_API_KEY")

    input_questions = [
        "What is the refund policy?",
        "How can I track my order?",
        "Can I change my delivery address after placing an order?"
    ]

    api_rows = []

    for question in input_questions:
        response = requests.post(
            api_url,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "question": question
            }
        )

        api_result = response.json()

        api_rows.append(
            {
                "question": question,
                "actual_output": api_result["answer"]
            }
        )

    qa_df = pd.DataFrame(api_rows)


elif source_type == "manual":
    manual_test_cases = [
        {
            "question": "What is the refund policy?",
            "actual_output": "Customers can request a refund within 15 days of purchase if the item is unused and in original condition."
        },
        {
            "question": "How can I track my order?",
            "actual_output": "You can track your order using the tracking link sent to your registered email or mobile number."
        },
        {
            "question": "Can I change my delivery address after placing an order?",
            "actual_output": "You can change the delivery address before the order is shipped by contacting customer support."
        }
    ]

    qa_df = pd.DataFrame(manual_test_cases)


display(qa_df.head())

,question,actual_output
0,What is the refund policy?,Customers can request a refund within 15 days ...
1,How can I track my order?,You can track your order using the tracking li...
2,Can I change my delivery address after placing...,You can change the delivery address before the...


In [5]:
required_columns = [
    "question",
    "actual_output"
]

missing_columns = []

for column in required_columns:
    if column not in qa_df.columns:
        missing_columns.append(column)

if len(missing_columns) == 0:
    print("All required columns are present.")
else:
    print("Missing columns:", missing_columns)

All required columns are present.


In [6]:
test_cases = []

for row_index, row in qa_df.iterrows():
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["actual_output"]
    )

    test_cases.append(test_case)

print("Total test cases created:", len(test_cases))

Total test cases created: 3


In [7]:
answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

In [8]:
evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=[answer_relevancy_metric]
)

print("Evaluation completed.")

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=3 | failed=0                 │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=150887;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.86s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Evaluation completed.
